# pipeline_v2 — crop IR 10.5 µm and projected LI-AFA into region folders

Changes relative to `pipeline.ipynb` (v1), and why:

1. **No per-crop stretching.** v1 saved crops with `plt.imsave(..., cmap='gray')`, which rescales every crop to its own min–max (verified: a crop with values 0–40 is saved as 0–255; an IR crop spanning 120–160 as 0–255). IR values therefore did not mean the same brightness temperature across crops, and LI was stretched a second time. v2 writes the pixel values unchanged with PIL.
2. **Lossless PNG** for both channels (v1: JPEG re-compression of every crop).
3. **LI source = `afa_projected_v2`** (fixed-scale AFA, `projection_rutine_v2.py`). v1 used the default `li_source='COUNT'` (per-pixel record-presence map), not the AFA values.
4. **Fast file matching** by timestamp lookup (v1 scanned all LI files for every IR file).

Unchanged: crop geometry (the coordinates in `LONLAT_DICT` are the crop's **upper-left corner**; the crop extends `crop_size` pixels east and south), file naming `{index:05}_{start}_{end}_{channel}.png`, per-crop `.wld`, and `li_labels.csv`. `split_dataset.py` works unchanged on the output; `dataset.py` (branch `dataretrain`) reads PNG.

In [ ]:
import csv, glob, os, re
from datetime import datetime
import numpy as np
from PIL import Image
from tqdm.notebook import tqdm

METEOSAT_ROOT_FOLDER = '/media/vladlanda/DATA/Meteosat'
IR_ROOT = os.path.join(METEOSAT_ROOT_FOLDER, 'ir_105')
LI_ROOT = os.path.join(METEOSAT_ROOT_FOLDER, 'afa_projected_v2')
SAVE_ROOT_FOLDER = 'full_v2'
UPTO = datetime(2026, 2, 11)
CROP_SIZE = 256

# Upper-left corner (lon, lat) of each crop -- NOT the region centre.
LONLAT_DICT = {
    'central_africa_1': (27.975000, 4.256660),
    'central_africa_2': (20.017528, 4.293710),
    'central_africa_3': (11.301556, 12.091834),
    'central_africa_4': (12.029032, 4.277996),
}

In [ ]:
PAIR_RE = re.compile(r'\d{8}T\d{6}Z_\d{8}T\d{6}Z')
TIME_FORMAT = '%Y%m%dT%H%M%SZ'


def read_wld(wld_path):
    """https://gdal.org/en/stable/drivers/raster/wld.html (x_luc/y_luc = centre of upper-left pixel)"""
    keys = ['x_size', 'rot_y', 'rot_x', 'y_size', 'x_luc', 'y_luc']
    with open(wld_path) as f:
        return {k: float(line.strip()) for k, line in zip(keys, f.readlines())}


def load_gray(path):
    """uint8 array exactly as stored (no rescaling)."""
    with Image.open(path) as im:
        return np.array(im.convert('L'))


def get_file_pairs(ir_root, li_root, upto=UPTO):
    """[{'ir', 'li', 'wld', 'time'}] matched by the IR start/end timestamp pair."""
    li_by_time = {}
    for f in glob.glob(os.path.join(li_root, '*', '*AFA*.png')):
        m = PAIR_RE.search(os.path.basename(f))
        if m:
            li_by_time[m.group(0)] = f
    pairs = []
    for ir in sorted(glob.glob(os.path.join(ir_root, '*', '*.jpg'))):
        m = PAIR_RE.search(os.path.basename(ir))
        if not m or datetime.strptime(m.group(0)[:16], TIME_FORMAT) > upto:
            continue
        wld = ir[:-4] + '.wld'
        li = li_by_time.get(m.group(0))
        if li is not None and os.path.isfile(wld):
            pairs.append({'ir': ir, 'li': li, 'wld': wld, 'time': m.group(0)})
    return pairs


def crop_window(wld, lon, lat, crop_size=CROP_SIZE):
    """Same geometry as v1: (lon, lat) is the upper-left corner of the crop."""
    left_edge = wld['x_luc'] - wld['x_size'] / 2
    top_edge = wld['y_luc'] - wld['y_size'] / 2
    dx = int(np.floor((lon - left_edge) / wld['x_size']))
    dy = int(np.floor((top_edge - lat) / -wld['y_size']))
    wld_text = (f"{wld['x_size']}\n{wld['rot_y']}\n{wld['rot_x']}\n{wld['y_size']}\n"
                f"{wld['x_luc'] + wld['x_size'] * dx}\n{wld['y_luc'] + wld['y_size'] * dy}\n")
    return dy, dx, wld_text


def crop_and_save_all(pairs, save_root, lonlat_dict, crop_size=CROP_SIZE):
    """Write {index:05}_{time}_{ir|li}.png + .wld per region; values unchanged."""
    labels = []
    for index, p in enumerate(tqdm(pairs)):
        try:
            wld = read_wld(p['wld'])
            images = {'ir': load_gray(p['ir']), 'li': load_gray(p['li'])}
        except OSError as e:
            print('skip', p['ir'], e)
            continue
        if images['ir'].shape != images['li'].shape:
            print('shape mismatch, skip', p['ir'], images['ir'].shape, images['li'].shape)
            continue
        for region, (lon, lat) in lonlat_dict.items():
            dy, dx, wld_text = crop_window(wld, lon, lat, crop_size)
            folder = os.path.join(save_root, region)
            os.makedirs(folder, exist_ok=True)
            stem = os.path.join(folder, f"{index:05}_{p['time']}")
            for ch, im in images.items():
                crop = im[dy:dy + crop_size, dx:dx + crop_size]
                if crop.shape != (crop_size, crop_size):
                    raise ValueError(f'{region}: crop {crop.shape} falls outside the image')
                Image.fromarray(crop, mode='L').save(f'{stem}_{ch}.png')
                if ch == 'li':
                    labels.append([region, os.path.basename(stem) + '_li.png', int((crop > 0).sum())])
            if not os.path.isfile(stem + '.wld'):
                with open(stem + '.wld', 'w') as f:
                    f.write(wld_text)
    with open(os.path.join(save_root, 'li_labels.csv'), 'w', newline='') as f:
        csv.writer(f).writerows(labels)
    return labels

In [ ]:
pairs = get_file_pairs(IR_ROOT, LI_ROOT)
print(len(pairs), 'IR/LI pairs;', pairs[:1])

In [ ]:
labels = crop_and_save_all(pairs, SAVE_ROOT_FOLDER, LONLAT_DICT)

## Verification: crops must equal the source pixels exactly (no stretching, lossless)

In [ ]:
p = pairs[len(pairs) // 2]
wld = read_wld(p['wld'])
for ch in ['ir', 'li']:
    src = load_gray(p[ch])
    for region, (lon, lat) in LONLAT_DICT.items():
        dy, dx, _ = crop_window(wld, lon, lat)
        index = pairs.index(p)
        saved = load_gray(os.path.join(SAVE_ROOT_FOLDER, region, f"{index:05}_{p['time']}_{ch}.png"))
        same = np.array_equal(saved, src[dy:dy + CROP_SIZE, dx:dx + CROP_SIZE])
        print(f'{ch} {region}: identical to source = {same} | min {saved.min()} max {saved.max()}')